# 02 — Weight of Evidence (WoE) & Information Value (IV)

**Goal:** turn categories into risk-ordered numbers (WoE) and rank features by how predictive they
are (IV). This is the last piece of Phase 1 theory and the direct input to the logistic-regression
scorecard we build next.

You'll build the WoE calculation **by hand for one feature first**, then generalize it into a
reusable function, then rank every feature. Two cells are `TODO (YOU)`.

## 1. Load data (now from our shared loader)

Last notebook we defined the schema inline. We've moved that into `src/data_loader.py` so every
notebook shares one source of truth — this is the "notebook code graduates into `src/`" pattern.
We add the repo root to the import path, then import the loader.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # so `src` is importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style="whitegrid")

from src.data_loader import load_german_credit, GERMAN_CATEGORICAL, GERMAN_NUMERIC

df = load_german_credit()          # decoded, with 0/1 `default` column
print("shape:", df.shape)
print("overall default rate:", round(df["default"].mean(), 3))
df[["checking_status", "credit_amount", "age_years", "default"]].head()

## 2. WoE by hand for one feature

Let's compute WoE for `checking_status` step by step so nothing is a black box.

**Vocabulary:** a *good* is a borrower who repaid (`default == 0`); a *bad* is one who defaulted
(`default == 1`). For each category we need: how many goods and bads it contains.

In [ ]:
feature = "checking_status"

tab = pd.DataFrame({
    "n":     df.groupby(feature)["default"].size(),
    "n_bad": df.groupby(feature)["default"].sum(),      # default==1 are the bads
})
tab["n_good"] = tab["n"] - tab["n_bad"]
tab

Now the two distributions. `dist_good` = what fraction of **all** goods live in this category;
`dist_bad` = what fraction of **all** bads live here. We add a small `0.5` to each count first —
that's the fix for the **zero-cell problem**: if a category had 0 bads, `dist_bad` would be 0 and
`ln(.../0)` would be infinite. The 0.5 nudge keeps it finite and barely changes non-zero cells.

In [ ]:
eps = 0.5
total_good = tab["n_good"].sum()
total_bad  = tab["n_bad"].sum()

tab["dist_good"] = (tab["n_good"] + eps) / total_good
tab["dist_bad"]  = (tab["n_bad"]  + eps) / total_bad

# WoE = ln(dist_good / dist_bad):  > 0 safer than average, < 0 riskier
tab["woe"] = np.log(tab["dist_good"] / tab["dist_bad"])
tab["default_rate"] = tab["n_bad"] / tab["n"]      # for sanity-checking the sign
tab.round(3)

Sanity check the signs against last notebook: `no account` had the *lowest* default rate, so
it should have the *highest* (most positive) WoE. `< 0 DM` had the highest default rate, so the most
negative WoE. Confirm that in the table above — WoE should move opposite to `default_rate`.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
w = tab["woe"].sort_values()
colors = ["#e76f51" if v < 0 else "#2a9d8f" for v in w.values]
ax.barh(w.index, w.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("WoE by checking_status  (green = safer, red = riskier)")
ax.set_xlabel("WoE")
plt.tight_layout()
plt.show()

## 3. Information Value for this feature

IV sums each category's contribution `(dist_good - dist_bad) * woe`. A category that is both large
and lopsided (very different good/bad shares) contributes a lot; a balanced category contributes ~0.

In [ ]:
tab["iv_component"] = (tab["dist_good"] - tab["dist_bad"]) * tab["woe"]
iv_checking = tab["iv_component"].sum()
print(tab[["iv_component"]].round(4).to_string())
print(f"\nIV(checking_status) = {iv_checking:.4f}")

Compare against the rule of thumb: <0.02 useless · 0.02-0.1 weak · 0.1-0.3 medium ·
0.3-0.5 strong · >0.5 suspiciously strong. Where does `checking_status` land?

## 4. Generalize into a reusable function

Same math, packaged. It returns the per-category WoE table **and** the single IV number.

In [ ]:
def woe_iv(df, feature, target="default", eps=0.5):
    """Compute the WoE table and total IV for one categorical `feature`."""
    grp = df.groupby(feature)[target]
    t = pd.DataFrame({"n": grp.size(), "n_bad": grp.sum()})
    t["n_good"] = t["n"] - t["n_bad"]
    t["dist_good"] = (t["n_good"] + eps) / t["n_good"].sum()
    t["dist_bad"]  = (t["n_bad"]  + eps) / t["n_bad"].sum()
    t["woe"] = np.log(t["dist_good"] / t["dist_bad"])
    t["iv_component"] = (t["dist_good"] - t["dist_bad"]) * t["woe"]
    iv = t["iv_component"].sum()
    return t, iv

# check it matches our by-hand result
_, iv_check = woe_iv(df, "checking_status")
print("function IV(checking_status) =", round(iv_check, 4), "(should match above)")

## 5. Rank every categorical feature by IV

This is feature selection in action: compute IV for all categoricals and sort. High IV = keep and
lean on; near-zero IV = probably drop.

In [ ]:
iv_scores = {}
for col in GERMAN_CATEGORICAL:
    _, iv = woe_iv(df, col)
    iv_scores[col] = iv

iv_series = pd.Series(iv_scores).sort_values(ascending=False)
print(iv_series.round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(iv_series.index[::-1], iv_series.values[::-1], color="#457b9d")
for x in [0.02, 0.1, 0.3]:
    ax.axvline(x, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Information Value by feature (dashed: 0.02 / 0.1 / 0.3 thresholds)")
ax.set_xlabel("IV")
plt.tight_layout()
plt.show()

**TODO (YOU):** read the ranking above and bucket the features. Using the rule of thumb, fill in
the three lists below with the feature names you'd call weak / medium / strong. (No single right
answer at the boundaries — the point is to practice reading IV.)

In [ ]:
# TODO (YOU): fill these lists based on iv_series and the rule-of-thumb thresholds
strong_features = []   # IV > 0.3
medium_features = []   # 0.1 - 0.3
weak_features   = []   # < 0.1

print("strong:", strong_features)
print("medium:", medium_features)
print("weak:  ", weak_features)

## 6. Numeric features need binning first

WoE works on groups. A numeric column like `age_years` has ~50 distinct values, so we first cut it
into a handful of ranges (bins), then treat each bin as a category. `pd.qcut` makes **quantile**
bins — each bin holds roughly the same number of borrowers.

In [ ]:
df["age_bin"] = pd.qcut(df["age_years"], q=5, duplicates="drop")
age_tab, age_iv = woe_iv(df, "age_bin")
print(age_tab[["n", "n_bad", "woe", "iv_component"]].round(3).to_string())
print(f"\nIV(age_years, 5 bins) = {age_iv:.4f}")

Notice WoE usually rises with age here: older borrowers tend to be safer. That monotonic trend
is exactly what a scorecard likes. (In real projects you sometimes enforce monotonic bins on purpose;
we'll keep quantile bins for now.)

**TODO (YOU):** do the same for another numeric feature. Bin it with `pd.qcut(..., q=5,
duplicates='drop')`, run `woe_iv`, and print the table + IV. Try `credit_amount` or `duration_months`.
Then judge: is it more or less predictive than `age_years`?

In [ ]:
# TODO (YOU): pick a numeric column, bin it, and compute its WoE table + IV
col = "..."                     # e.g. "credit_amount"
# df[col + "_bin"] = pd.qcut(df[col], q=5, duplicates="drop")
# tab_num, iv_num = woe_iv(df, col + "_bin")
# print(tab_num[["n", "n_bad", "woe", "iv_component"]].round(3).to_string())
# print(f"IV({col}) = {iv_num:.4f}")

## 7. WoE *transform*: encode a feature for modeling

Ranking is nice, but the payoff is **replacing categories with their WoE values** so a model can use
them. Here we map `checking_status` to a new numeric column `checking_status_woe`. Next session,
columns like this become the inputs to logistic regression, and the model's coefficients turn into
scorecard points.

In [ ]:
woe_map = tab["woe"].to_dict()      # from section 2 (checking_status)
df["checking_status_woe"] = df["checking_status"].map(woe_map)
df[["checking_status", "checking_status_woe", "default"]].head(8)

## 8. Recap — what you learned

- **WoE** turns each category into `ln(good_share / bad_share)`: positive = safer, negative = riskier,
  zero = uninformative. It moves opposite to the default rate.
- The **zero-cell problem** (log of zero) is handled with a small `+0.5` count adjustment.
- **IV** = `sum((good_share - bad_share) * woe)` scores a whole feature's predictive power; the
  <0.02 / 0.1 / 0.3 / 0.5 rule of thumb buckets features from useless to suspiciously strong.
- **Numeric features** are binned (e.g. `pd.qcut` quantiles) before WoE.
- **WoE transform** replaces categories with their WoE number, producing model-ready columns — the
  bridge to the logistic-regression scorecard.

**Next session:** build the logistic-regression scorecard on German Credit — WoE-encode all features,
fit logistic regression, and convert coefficients into integer point scores (the format lenders
actually use). Then bring in *Give Me Some Credit* from Kaggle.